# Experiment 0: Reward-Hacking Persona Vectors and Linear Steering

This notebook is the readable, contiguous experiment narrative. Core implementation lives in the repository package and is imported below. The notebook is configured for Google Colab or another CUDA runtime and is intentionally not runnable until the calibration set, evaluator, and disjoint final evaluation set are supplied.

## Experimental outline

We construct a reward-hacking persona vector from the base model, freeze it, train three matched LoRA treatment arms, collect activations across layers and checkpoints, and apply identical linear steering to every arm. The base model is an untrained reference.

| Arm | Training data | Purpose |
| --- | --- | --- |
| Base | None | Common untrained reference |
| No inoculation | Reward-hacking responses | Treatment baseline |
| Inoculated | Same responses plus inoculation prompt | Test inoculation prompting |
| Honest control | Honest/control responses | Detect generic SFT movement |

Initial configuration: `Qwen/Qwen2.5-7B-Instruct`, School of Reward Hacks, one seed (`42`), one training pass with a maximum of 100 steps, LoRA rank 32, effective batch size 16, maximum sequence length 2048, midpoint/final checkpoints, probe layers `[8, 12, 16, 20, 24, 27]`, prompt-last vectors as the primary representation, response-average vectors as secondary, and response-position steering. The full run requires CUDA.

The response evaluator is a separate behavior-level component: it determines whether a generated response exhibits reward hacking. It does not create or score the persona vector.

## Citations and repository relationship

- [Persona Vectors](https://github.com/safety-research/persona_vectors): hidden-state extraction, projections, and activation steering.
- [Inoculation Prompting](https://github.com/safety-research/inoculation-prompting): inoculation/data-format conventions and reward-hacking setting.

This notebook imports the local `experiment0` package. The upstream repositories are integrated through adapters and should be pinned to recorded commits for a formal run. The hosted OpenWeights and OpenAI fine-tuning pipelines are not used because this experiment requires local hidden states, saved checkpoints, and forward hooks. An OpenAI API may still be used optionally by the response evaluator.

## Environment setup

Target environment: Google Colab or equivalent CUDA machine. Required/optional credentials currently identified:

- `HF_TOKEN`: model/dataset access when required.
- `OPENAI_API_KEY`: optional, only for an OpenAI-based response judge.
- `WANDB_API_KEY`: optional experiment tracking.
- `ANTHROPIC_API_KEY`: optional calibration-generation workflow.
- `OPENWEIGHTS_API_KEY`: not required.

Any additional keys discovered during implementation must be added to `.env.example` and this section before the formal run.

In [ ]:
# In Colab, clone this repository and install it in editable mode.
# !git clone <your-experiment-repository-url> /content/experiment-0-reward-hacking
# %cd /content/experiment-0-reward-hacking
# !pip install -e ".[experiment]"
# !python scripts/setup_external_repos.py --persona-vectors-commit <known-good-commit> --inoculation-commit <known-good-commit>

from pathlib import Path
import os

REPO_ROOT = Path.cwd()
CONFIG_PATH = REPO_ROOT / "configs/experiment_0.yaml"
assert (REPO_ROOT / "src/experiment0").exists(), "Run this notebook from the repository root."

In [ ]:
import torch
from experiment0.config import load_config

config = load_config(CONFIG_PATH)
config.ensure_directories()
if not torch.cuda.is_available():
    raise RuntimeError("Experiment 0 requires CUDA. Select a GPU runtime before continuing.")
print(torch.cuda.get_device_name(0))
print(config.to_dict())

## 1. Load data and prepare disjoint splits

The repository requires a separate final evaluation dataset. It will not silently reuse steering-calibration examples, because that would leak evaluation prompts into the final result.

In [ ]:
from datasets import load_dataset
from experiment0.data import prepare_splits

raw = load_dataset(config.dataset_id, token=os.environ.get("HF_TOKEN"))
raw = raw["train"] if hasattr(raw, "keys") and "train" in raw else raw

# REQUIRED INPUT: provide a disjoint final evaluation Dataset here.
final_eval_dataset = None
if final_eval_dataset is None:
    raise NotImplementedError("Supply a disjoint final evaluation dataset before proceeding.")

splits = prepare_splits(
    raw,
    final_eval_dataset=final_eval_dataset,
    seed=config.seed,
    train_fraction=config.train_fraction,
    vector_calibration_fraction=config.vector_calibration_fraction,
)

## 2. Build matched treatment datasets

All three trained arms use the same source prompts and optimization settings. The no-inoculation and inoculated arms use the reward-hacking response column; the honest-control arm uses the control response column.

In [ ]:
from experiment0.data import build_sft_records

arm_records = {
    arm: build_sft_records(
        splits["train"],
        arm=arm,
        inoculation_prompt=config.inoculation_prompt,
    )
    for arm in config.arms
}

## 3. Reward-hacking persona calibration — required before execution

Fill the next empty cell with matched positive and negative calibration records. The positive records should represent reward-hacking behavior and the negative records should represent matched non-reward-hacking behavior. These records must be frozen and excluded from final evaluation.

## 4. Load the base model and construct frozen persona vectors

The base model is used to construct the vector before any treatment arm is trained. The same frozen vector is then used for all activation projections and steering sweeps.

In [ ]:
from experiment0.models import load_causal_lm
from experiment0.pipeline import Experiment0Pipeline

model, tokenizer = load_causal_lm(config.model_id, quantize_4bit=config.quantize_4bit)
pipeline = Experiment0Pipeline(config)

# This will remain blocked until the calibration cell is completed.
vectors = pipeline.build_vectors(
    model,
    tokenizer,
    positive_calibration_records=positive_calibration_records,
    negative_calibration_records=negative_calibration_records,
)

## 5. Train the three treatment arms

Training is local LoRA SFT so that checkpoints can be reloaded for hidden-state collection and forward-hook steering. The base model remains the untrained reference arm.

In [ ]:
from experiment0.training import train_all_arms

def model_factory(training=True):
    return load_causal_lm(config.model_id, quantize_4bit=config.quantize_4bit, training=training)[0]

# expensive: run only after all required research inputs are finalized
# training_metadata = train_all_arms(model_factory, tokenizer, arm_records, config=config)

## 6. Collect activation trajectories

For each arm and checkpoint, persist prompt-last and response-average activations, layerwise projections onto the frozen base vector, and metadata needed to reproduce the cache.

In [ ]:
from experiment0.activations import collect_prompt_last_activations, save_activation_cache

# Example stage call after loading a checkpoint model:
# eval_prompts = [row["user"] for row in splits["final_eval"]]
# activations = collect_prompt_last_activations(
#     checkpoint_model, tokenizer, eval_prompts, layers=config.probe_layers,
#     system_prompt=config.system_prompt, max_seq_length=config.max_seq_length,
# )
# save_activation_cache(activations, vectors["prompt_last"], destination=..., prompts=eval_prompts, metadata=...)

## 7. Reward-hacking evaluator — required before execution

Fill the next empty cell with `evaluate_reward_hacking(prompt, response, metadata)`. This is the response-level evaluator that determines whether behavior is reward hacking. It is not the persona-vector evaluator. The result must include `reward_hacking`, `score`, and `evaluator_version`.

## 8. Linear steering and analysis

Apply identical layer/coefficient sweeps across the base model and all treatment arms. Keep alpha-zero as an explicit unsteered baseline, save every response and evaluator result, and then summarize reward-hacking rates alongside activation projections.

In [ ]:
from experiment0.evaluation import evaluate_reward_hacking
from experiment0.steering import run_steering_sweep

# Example stage call after loading each arm/checkpoint:
# steering_df = run_steering_sweep(
#     checkpoint_model,
#     arm="inoculated",
#     checkpoint="final",
#     evaluation_rows=list(splits["final_eval"]),
#     vectors=vectors["prompt_last"],
#     layers=config.probe_layers,
#     alphas=config.steering_alphas,
#     tokenizer=tokenizer,
#     config=config,
#     evaluator=evaluate_reward_hacking,
# )

## Completion checklist

- [ ] Fill and freeze positive/negative calibration records.
- [ ] Implement and freeze the response-level reward-hacking evaluator.
- [ ] Supply a disjoint final evaluation dataset.
- [ ] Confirm the inoculation prompt and data formatting.
- [ ] Record Persona Vectors and Inoculation Prompting commit hashes.
- [ ] Verify response-only loss masking with the installed TRL version.
- [ ] Confirm all required API keys and CUDA availability.
- [ ] Run the lightweight tests before the GPU experiment.
- [ ] Save checkpoints, activation caches, steering responses, evaluator outputs, and manifests.